# nteract/desktop: Strategic Alignment with Anaconda Themes

This document maps nteract/desktop's current capabilities to strategic themes discussed in the Anaconda alignment meeting.

---

## 1. Environment Flexibility (Beyond Conda-Only)

**Meeting theme:** Jack expressed interest in moving beyond Conda as the sole option.

**nteract/desktop capabilities:**

| Backend | Project File | Status |
|---------|--------------|--------|
| UV | `pyproject.toml` | Fully supported |
| Conda | `environment.yml` | Fully supported |
| Pixi | `pixi.toml` | Fully supported (conda + PyPI) |
| Deno | `deno.json` | Fully supported |

**Project file detection:** "Closest wins" algorithm walks up from notebook directory, stops at `.git` boundaries. Enables per-project environments without global config.

**Inline dependencies:** Notebooks can embed deps in metadata (`metadata.runt.uv` or `metadata.runt.conda`). Cached in `~/.cache/runt/inline-envs/` keyed by dependency hash. Same deps = instant reuse.

**Relevance:** Already delivers the multi-backend flexibility Anaconda wants. Could serve as reference implementation for workspace environment strategy.

---

## 2. Real-Time Sync & Local-First Architecture

**Meeting theme:** Kyle highlighted real-time document syncing as a key differentiator.

**Architecture:**

```
Frontend (WASM)  <--Automerge sync-->  Daemon (runtimed)  <--Automerge sync-->  Other windows
     |                                       |
     v                                       v
  Instant local                         Kernel execution
  cell mutations                        Output persistence
```

**Key properties:**
- **Instant feedback:** Cell edits apply locally in WASM before sync (optimistic updates)
- **Automatic conflict resolution:** Automerge CRDT merges concurrent edits at character level
- **Multi-window:** Same notebook open in multiple windows syncs in real-time
- **Offline-capable:** Local doc can operate without daemon (limited functionality)

**Persistence:**
- Saved notebooks: `.ipynb` is source of truth
- Untitled notebooks: Automerge doc in `~/.cache/runt/notebook-docs/` survives daemon restarts

**Relevance:** Demonstrates a production-quality local-first architecture that could inform Anaconda's workspace sync strategy.

---

## 3. Secure Package Management

**Meeting theme:** Jack emphasized "secure by default" for paid tiers, with visibility into security issues on free tiers.

**Trust system implemented:**

| Component | Implementation |
|-----------|----------------|
| Key storage | `~/.config/runt/trust-key` (per-machine, 32-byte HMAC key) |
| Signature | HMAC-SHA256 over canonical JSON of dependency metadata |
| Scope | Signs `metadata.runt.uv` and `metadata.runt.conda` only |
| Status values | `Trusted`, `Untrusted`, `SignatureInvalid`, `NoDependencies` |

**Behavior:**
- Notebooks from others arrive `Untrusted` (key is machine-specific)
- User must explicitly trust before auto-installing dependencies
- Editing cell code doesn't invalidate trust (only dependency changes do)

**Extension opportunities:**
- Add vulnerability scanning integration (e.g., Anaconda's security feed)
- Surface trust status in UI with remediation guidance
- Support org-level trust keys for team notebooks

**Relevance:** Foundation exists for secure-by-default. Could extend with Anaconda's security tooling.

---

## 4. Agent/Sub-Agent Tooling for EDA

**Meeting theme:** Kyle discussed agents spawning sub-agents with their own runtimes for exploratory data analysis.

**runtimed-py bindings (ready now):**

```python
import runtimed

# Agent creates isolated notebook session
session = runtimed.Session(notebook_id="eda-task-1")
session.start_kernel(kernel_type="python", env_source="auto")

# Execute code and get results
result = session.run("df.describe()", timeout_secs=60)
print(result.stdout)
print(result.outputs)  # Display data, images, etc.

# Stream execution for real-time monitoring
for event in session.stream_execute(cell_id):
    if event.event_type == "output":
        process_output(event.output)

# Manage dependencies programmatically
session.add_uv_dependency("pandas>=2.0")
```

**MCP server (available now via `nteract` on PyPI):**

The MCP server has been implemented in the main nteract repo (https://github.com/nteract/nteract) and provides:
- Session management: `connect_notebook`, `list_notebooks`
- Kernel control: `start_kernel`, `shutdown_kernel`, `interrupt_kernel`
- Cell operations: `create_cell`, `execute_cell`, `get_all_cells`

**Architecture advantage:** Multiple agents can connect to the same `notebook_id` and see each other's changes via Automerge sync. Enables agent collaboration patterns.

**Relevance:** Both Python bindings and MCP server are production-ready. MCP server exposes notebook tooling to any MCP-compatible agent (Claude, Copilot extensions, etc.).

---

## 5. Standard Protocol for Notebooks

**Meeting theme:** Kyle emphasized the need for a standard protocol so users don't leave the desktop app for rendering/editing.

**Daemon protocol (wire format):**

```
Frame: [4-byte length] [1-byte type] [payload]

Types:
  0x00: AutomergeSync (raw CRDT sync messages)
  0x01: NotebookRequest (JSON RPC-style)
  0x02: NotebookResponse (JSON)
  0x03: NotebookBroadcast (kernel events, outputs)
```

**Daemon socket:** `~/.cache/runt/runtimed.sock`

**What can connect:**
- nteract/desktop (Tauri app)
- runtimed-py (Python agents)
- Any Unix socket client speaking the protocol

**Extensibility:** The protocol separates sync (Automerge) from execution (broadcasts). New clients can implement just what they need.

**Relevance:** Could evolve into a documented standard for notebook runtime interop. Other tools could connect to the same daemon.

---

## 6. Workspace-Centered View

**Meeting theme:** Workspaces as key focus for improving reliability and scalability.

**Current per-notebook state:**
- Environment detection tied to notebook's directory
- Project file walk-up scoped by `.git` boundary
- Inline deps scoped to individual notebook

**Potential workspace extensions:**
1. **Workspace-level environment config:** Override project file detection at workspace root
2. **Shared kernel pools:** Workspace-scoped prewarmed environments
3. **Workspace trust:** Sign all notebooks in a workspace with one action
4. **Workspace sync:** Daemon could manage workspace-level state beyond individual notebooks

**Relevance:** Architecture is notebook-centric today but could extend to workspace scope. The daemon + Automerge pattern generalizes.

---

## Summary: Alignment Matrix

| Anaconda Theme | nteract/desktop Status | Gap |
|----------------|----------------------|-----|
| Beyond Conda-only | UV, Conda, Pixi, Deno supported | None |
| Real-time sync | Automerge CRDT, multi-window | None |
| Secure packages | HMAC trust system | Vulnerability scanning |
| Agent tooling | runtimed-py + MCP server (`nteract` PyPI) | None |
| Standard protocol | Daemon wire protocol exists | Documentation |
| Workspace view | Notebook-centric | Workspace-level features |

---

## Suggested Next Steps

1. **For workshops:** Use this doc to show where nteract/desktop already aligns
2. **Security extension:** Add Anaconda security feed integration to trust system
3. **Workspace features:** Design workspace-level config that layers over notebook detection
4. **Agent demos:** Showcase MCP server (`nteract` on PyPI) integration with Claude/Copilot
